# <center>**Chapter 20 - Optimize Hyperparameters with Grid Search**<center>

## **Mục tiêu**
Dùng `GridSearchCV` để so sánh các lựa chọn về optimizer, learning rate, momentum, batch size, số epoch, kích thước lớp ẩn, activation, dropout và khởi tạo trọng số. Mô hình được triển khai bằng PyTorch và bọc với skorch.

Thí nghiệm dùng bộ Pima Indians Diabetes có sẵn trong `code/data`. Chia riêng tập test trước khi tìm kiếm; điền giá trị thiếu và chuẩn hóa nằm trong `Pipeline`, nhờ đó mỗi fold chỉ học các bước tiền xử lý từ dữ liệu huấn luyện của fold đó. Hai lượt tìm kiếm nhỏ theo từng nhóm giúp thời gian chạy phù hợp với buổi thực hành.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetClassifier
from torch import nn, optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_pima_file() -> Path:
    # Tìm tệp dữ liệu từ thư mục dự án hoặc thư mục notebook.
    lab_dir_name = "Buổi thực hành số 7 (Optimizing DL Networks)"
    current = Path.cwd().resolve()
    for root in (current, *current.parents):
        candidates = (
            root / "data" / "pima-indians-diabetes.csv",
            root / "code" / "data" / "pima-indians-diabetes.csv",
            root / lab_dir_name / "code" / "data" / "pima-indians-diabetes.csv",
        )
        for candidate in candidates:
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Không tìm thấy pima-indians-diabetes.csv trong code/data.")

In [2]:
pima = pd.read_csv(find_pima_file())
X_frame = pima.iloc[:, :-1].copy()
y = pima.iloc[:, -1].to_numpy(dtype=np.int64)

# Các số 0 ở một số chỉ số y tế biểu diễn giá trị thiếu; số lần mang thai vẫn giữ nguyên.
columns_with_missing_zero = X_frame.columns[[1, 2, 3, 4, 5]]
X_frame[columns_with_missing_zero] = X_frame[columns_with_missing_zero].astype(float).replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X_frame.to_numpy(dtype=np.float32),
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
print(f"Tập huấn luyện: {X_train.shape}; tập test giữ riêng: {X_test.shape}")
print("Thiết bị:", DEVICE)

Tập huấn luyện: (614, 8); tập test giữ riêng: (154, 8)
Thiết bị: cpu


## **Định nghĩa mạng và Pipeline**

Lớp cuối trả về hai logits, phù hợp với `CrossEntropyLoss`. Pipeline bảo đảm imputer và scaler được fit riêng trong mỗi fold. Các lưới được chia thành hai giai đoạn để thực hành theo nhóm siêu tham số; đây không phải tìm kiếm toàn cục trên mọi tích Descartes.

In [5]:
class PimaClassifier(nn.Module):
    def __init__(
        self,
        hidden_size: int = 16,
        activation=nn.ReLU,
        dropout_rate: float = 0.1,
        weight_init=nn.init.xavier_uniform_,
    ):
        super().__init__()
        self.hidden = nn.Linear(8, hidden_size)
        self.activation = activation()
        self.dropout = nn.Dropout(p=dropout_rate)
        self.output = nn.Linear(hidden_size, 2)

        # Dùng cùng phương pháp khởi tạo cho cả hai lớp tuyến tính.
        for layer in (self.hidden, self.output):
            weight_init(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        hidden = self.dropout(self.activation(self.hidden(features)))
        return self.output(hidden)

model = NeuralNetClassifier(
    module=PimaClassifier,
    criterion=nn.CrossEntropyLoss,
    optimizer=optim.Adam,
    lr=1e-2,
    max_epochs=10,
    batch_size=32,
    iterator_train__shuffle=True,
    train_split=None,
    device=DEVICE,
    verbose=0,
)
pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("net", model),
    ]
)

# Lượt 1: so sánh optimizer, learning rate, momentum và batch size.
training_grid = [
    {
        "net__optimizer": [optim.Adam],
        "net__lr": [1e-3, 1e-2],
        "net__batch_size": [16, 32],
        "net__max_epochs": [8, 12],
    },
    {
        "net__optimizer": [optim.SGD],
        "net__lr": [1e-2],
        "net__optimizer__momentum": [0.0, 0.9],
        "net__batch_size": [16, 32],
        "net__max_epochs": [8, 12],
    },
]
training_search = GridSearchCV(
    pipeline,
    param_grid=training_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,
    refit=True,
    error_score="raise",
    return_train_score=True,
)
training_search.fit(X_train, y_train)
print(f"Điểm CV tốt nhất lượt 1: {training_search.best_score_:.3f}")
print("Tham số tốt nhất lượt 1:", training_search.best_params_)

Điểm CV tốt nhất lượt 1: 0.782
Tham số tốt nhất lượt 1: {'net__batch_size': 32, 'net__lr': 0.01, 'net__max_epochs': 8, 'net__optimizer': <class 'torch.optim.adam.Adam'>}


In [6]:
# Lượt 2: tinh chỉnh kiến trúc, activation, dropout và khởi tạo trọng số.
architecture_grid = {
    "net__module__hidden_size": [8, 16],
    "net__module__activation": [nn.ReLU, nn.Tanh],
    "net__module__dropout_rate": [0.0, 0.25],
    "net__module__weight_init": [
        nn.init.xavier_uniform_,
        nn.init.kaiming_uniform_,
    ],
}
architecture_search = GridSearchCV(
    training_search.best_estimator_,
    param_grid=architecture_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,
    refit=True,
    error_score="raise",
    return_train_score=True,
)

architecture_search.fit(X_train, y_train)
print(f"Điểm CV tốt nhất lượt 2: {architecture_search.best_score_:.3f}")
print("Tham số tốt nhất lượt 2:", architecture_search.best_params_)

# Chỉ đánh giá tập test sau khi hoàn tất lựa chọn siêu tham số.
test_predictions = architecture_search.best_estimator_.predict(X_test)
print(f"Accuracy trên tập test giữ riêng: {accuracy_score(y_test, test_predictions):.3f}")

Điểm CV tốt nhất lượt 2: 0.788
Tham số tốt nhất lượt 2: {'net__module__activation': <class 'torch.nn.modules.activation.Tanh'>, 'net__module__dropout_rate': 0.25, 'net__module__hidden_size': 8, 'net__module__weight_init': <function kaiming_uniform_ at 0x00000245225D49A0>}
Accuracy trên tập test giữ riêng: 0.688


## **Kết luận**

`GridSearchCV` chọn cấu hình dựa trên cross-validation; `best_estimator_` được refit trên toàn bộ tập train. Tập test chỉ được dùng để đánh giá cuối, không dùng để chọn cấu hình. Lưới nhỏ minh họa quy trình; kết quả phụ thuộc dữ liệu và không nên chuyển nguyên xi sang bài toán khác.

# <center>**HẾT**<center>